# NB09 — Multi-Metric Weighted Scoring

**Purpose:** Combine all individual metric CSVs (nb03–nb08) into a composite score.

**Experiments:** Equal-weight, physics-informed, grid-search (in-sample), LOOCV-optimised (honest)

**Runs on:** Local CPU

## 0. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pydicom', 'numpy', 'pandas', 'matplotlib', 'Pillow', 'openpyxl', 'scipy'])
print('Dependencies OK')

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
from itertools import product
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0f0f1a'
matplotlib.rcParams['axes.facecolor']   = '#1a1a2e'
matplotlib.rcParams['text.color']       = 'white'
%matplotlib inline

GT_PATH = r'E:/CINE-Test1/1stcohortSAX (1).xlsx'
OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
EXCLUDE = ['Patient-KP']
OFFSETS_13 = [-150 + i*25 for i in range(13)]

SIGMA_CENTER = 100.0
def gaussian_center_prior(hz_array):
    g = np.exp(-0.5 * (np.array(hz_array) / SIGMA_CENTER)**2)
    mn, mx = g.min(), g.max()
    return (g - mn) / (mx - mn) if mx > mn else g

print('Config OK')

## 1. Load All Metric CSVs

In [ ]:
from utils.gt_loader import load_gt

gt_dict = load_gt(GT_PATH)

df_signal   = pd.read_csv(f'{OUT_DIR}/signal_metrics.csv')
df_contrast = pd.read_csv(f'{OUT_DIR}/contrast_metrics.csv')
df_banding  = pd.read_csv(f'{OUT_DIR}/banding_metrics.csv')
df_edge     = pd.read_csv(f'{OUT_DIR}/edge_metrics.csv')
df_temporal = pd.read_csv(f'{OUT_DIR}/temporal_metrics.csv')

cnn_path = f'{OUT_DIR}/cnn_scores.csv'
if os.path.exists(cnn_path):
    df_cnn  = pd.read_csv(cnn_path)
    has_cnn = True
    print('CNN scores loaded.')
else:
    has_cnn = False
    print('WARNING: cnn_scores.csv not found — CNN component excluded. Run nb08 on Kaggle first.')

KEY = ['patient_id', 'image_no']
master = df_signal[KEY + ['offset_hz', 'mu_myo', 'snr_myo', 'homogeneity']].copy()
master = master.merge(df_contrast[KEY + ['cnr', 'contrast_ratio']], on=KEY)
master = master.merge(df_banding[KEY  + ['banded_fraction_myo', 'banding_burden', 'band_trough_depth']], on=KEY)
master = master.merge(df_edge[KEY     + ['edge_sharpness_boundary', 'edge_sharpness_myo', 'laplacian_variance']], on=KEY)
master = master.merge(df_temporal[KEY + ['temporal_std_sweep', 'local_signal_drop']], on=KEY)
if has_cnn:
    master = master.merge(df_cnn[KEY  + ['cnn_score']], on=KEY, how='left')
    master['cnn_score'] = master['cnn_score'].fillna(0.5)
else:
    master['cnn_score'] = 0.5

master = master[~master['patient_id'].isin(EXCLUDE)].copy()
print(f'Master: {len(master)} rows, {master["patient_id"].nunique()} patients')
master.head()

## 2. Normalise & Composite Functions

In [ ]:
def minmax_per_patient(df, col):
    def _norm(x):
        r = x.max() - x.min()
        return (x - x.min()) / r if r > 1e-9 else x * 0 + 0.5
    return df.groupby('patient_id')[col].transform(_norm)

master['n_mu_myo']              = minmax_per_patient(master, 'mu_myo')
master['n_cnr']                 = minmax_per_patient(master, 'cnr')
master['n_banded_fraction_myo'] = minmax_per_patient(master, 'banded_fraction_myo')
master['n_edge_boundary']       = minmax_per_patient(master, 'edge_sharpness_boundary')
master['n_temporal_std']        = minmax_per_patient(master, 'temporal_std_sweep')
master['n_cnn_score']           = minmax_per_patient(master, 'cnn_score')
master['center_prior']          = master.groupby('patient_id')['offset_hz'].transform(
    lambda hz: pd.Series(gaussian_center_prior(hz.values), index=hz.index)
)
print('Normalisation done.')

from utils.gt_loader import evaluate_predictions
from utils.vis_utils import summary_text

def compute_composite(df, w):
    score = (
        w[0] * df['n_mu_myo']
      + w[1] * df['n_cnr']
      + w[2] * (1 - df['n_banded_fraction_myo'])
      + w[3] * df['n_edge_boundary']
      + w[4] * (1 - df['n_temporal_std'])
      + w[5] * df['n_cnn_score']
      + w[6] * df['center_prior']
    )
    df = df.copy()
    df['composite_score'] = score
    return df

def evaluate_composite(df, gt_dict, weights):
    df_s = compute_composite(df, weights)
    pred_dict = {}
    for pid, grp in df_s.groupby('patient_id'):
        best_idx = int(np.argmax(grp.sort_values('image_no')['composite_score'].values))
        pred_dict[pid] = best_idx + 1
    return evaluate_predictions(pred_dict, gt_dict, exclude=EXCLUDE)

print('Composite functions defined.')

## 3. Experiments A–D: Equal / Physics / Grid-Search / LOOCV

In [ ]:
# A: Equal weight
W_EQUAL = [1/7]*7
res_equal = evaluate_composite(master, gt_dict, W_EQUAL)
print(summary_text('Equal-weight composite (1/7)', res_equal['top1'], res_equal['pm1'], res_equal['mae_hz'], res_equal['n_valid']))

# B: Physics-informed
W_PHYSICS = [0.05, 0.05, 0.55, 0.10, 0.05, 0.10, 0.10]
res_physics = evaluate_composite(master, gt_dict, W_PHYSICS)
print(summary_text('Physics-informed weights', res_physics['top1'], res_physics['pm1'], res_physics['mae_hz'], res_physics['n_valid']))

In [ ]:
# C: Coarse grid search (in-sample)
print('Running coarse grid search...')
GRID_VALS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
W_CENTER_FIXED = 0.10

best_top1 = 0.0; best_pm1 = 0.0; best_w = W_EQUAL; n_searched = 0

for w1, w2, w3, w4, w5, w6 in product(GRID_VALS, repeat=6):
    w_total = w1+w2+w3+w4+w5+w6+W_CENTER_FIXED
    if w_total < 0.5: continue
    weights = [x/w_total for x in [w1,w2,w3,w4,w5,w6,W_CENTER_FIXED]]
    res = evaluate_composite(master, gt_dict, weights)
    n_searched += 1
    if res['top1'] > best_top1 or (res['top1']==best_top1 and res['pm1']>best_pm1):
        best_top1, best_pm1, best_w = res['top1'], res['pm1'], weights[:]

print(f'Searched {n_searched} combinations.')
res_grid = evaluate_composite(master, gt_dict, best_w)
print(summary_text('Grid-search (IN-SAMPLE)', res_grid['top1'], res_grid['pm1'], res_grid['mae_hz'], res_grid['n_valid']))
labels = ['signal','cnr','no_bfm','edge','no_temp','cnn','center']
for lbl,w in zip(labels, best_w): print(f'  {lbl:15s}: {w:.3f}')

In [ ]:
# D: LOOCV weight search (honest)
print('Running LOOCV weight search (honest)...')
patient_ids = sorted(master['patient_id'].unique())
loocv_preds = {}

for test_pid in patient_ids:
    train_df = master[master['patient_id'] != test_pid].copy()
    test_df  = master[master['patient_id'] == test_pid].copy()
    best_top1_fold = 0.0; best_w_fold = [1/7]*7

    for w1, w2, w3, w4, w5, w6 in product(GRID_VALS, repeat=6):
        w_total = w1+w2+w3+w4+w5+w6+W_CENTER_FIXED
        if w_total < 0.5: continue
        weights = [x/w_total for x in [w1,w2,w3,w4,w5,w6,W_CENTER_FIXED]]
        res_train = evaluate_composite(train_df, gt_dict, weights)
        if res_train['top1'] > best_top1_fold:
            best_top1_fold, best_w_fold = res_train['top1'], weights[:]

    test_scored = compute_composite(test_df, best_w_fold)
    best_idx = int(np.argmax(test_scored.sort_values('image_no')['composite_score'].values))
    loocv_preds[test_pid] = best_idx + 1
    print(f'  {test_pid}: pred=#{best_idx+1}, gt=#{gt_dict[test_pid]["image_no"]}')

res_loocv = evaluate_predictions(loocv_preds, gt_dict, exclude=EXCLUDE)
print(summary_text('LOOCV-optimised composite (HONEST)', res_loocv['top1'], res_loocv['pm1'], res_loocv['mae_hz'], res_loocv['n_valid']))

## 4. Save Composite CSVs for nb10

In [ ]:
for w_name, w_vals in [('physics', W_PHYSICS), ('gridsearch', best_w), ('equal', W_EQUAL)]:
    df_out = compute_composite(master, w_vals)
    df_out[['patient_id','image_no','offset_hz','composite_score']].to_csv(
        f'{OUT_DIR}/composite_{w_name}.csv', index=False)
    print(f'Saved composite_{w_name}.csv')

print('NB09 complete. Proceed to nb10_evaluation.ipynb')